<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/seq2one/stage_06_01_logistic_regression_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06_01 - T2 SEQ2ONE - Baseline**

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [1]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-21 20:37:39,106 | INFO | Environment initialized


## **2. Acceso a drive**

In [2]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

2026-04-21 20:38:19,184 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


Mounted at /content/drive


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [3]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
  "t2_p40_h30",
  "t2_p40_h60",
  "t2_p50_h30",
]

# Tamaños de ventana
WINDOW_SIZES = [30]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
        "ema_60",
        "roc_60",
        "roc_30",
        "stoch_k_30",
        "mom_5",
        "atr_norm_10",
        "macd"
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-04-21 20:38:19,714 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-21 20:38:19,716 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-21 20:38:19,717 | INFO | Configuración de experimento cargada
2026-04-21 20:38:19,718 | INFO | Targets: ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
2026-04-21 20:38:19,719 | INFO | Window sizes: [30]


In [4]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_mnq_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:10]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[30]["t2_p40_h30"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-04-21 20:38:19,731 | INFO | Paths construidos (windows + scaler compartido T2)
2026-04-21 20:38:20,124 | INFO | Windows OK      : 9
2026-04-21 20:38:20,125 | INFO | Windows missing : 0
2026-04-21 20:38:20,126 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl
2026-04-21 20:38:20,127 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [5]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [6]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [7]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_p40_h30'
        - 't2_p40_h60'
        - 't2_p50_h30'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon_str = target.split("_")[-1]   # ej: "h30"
        horizon = int(horizon_str.replace("h", ""))
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }

### **4.4. Creación de bundles T2**

In [8]:
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundles : dict
        bundles[target] -> bundle dict
    """

    bundles = {}

    # --------------------------
    # Construcción
    # --------------------------
    for target in targets:
        bundles[target] = load_windows_and_scaler(
            window_size=window_size,
            target=target,
            windows_paths=windows_paths,
            scaler_path=scaler_path,
        )

    # --------------------------
    # Verificación rápida
    # --------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    for target in targets:
        b = bundles[target]

        print(f"\nTARGET: {target}")
        print("Train :", b["train"]["X"].shape, b["train"]["y"].shape)
        print("Valid :", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print("Test  :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print("Scaler:", type(b["scaler"]).__name__)

    return bundles

In [9]:
bundles_L30 = create_bundles(window_size=30)

2026-04-21 20:38:20,587 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-21 20:38:20,588 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-21 20:38:20,799 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-21 20:38:20,799 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-21 20:38:21,015 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-21 20:38:21,016 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-21 20:38:22,391 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-21 20:38:22,391 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-21 20:38:22,947 | INFO | Loaded: windows_t2_p40_h60_train.npz
2026-04-21 20:38:22,948 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-21 20:38:23,164 | INFO | Loaded: windows_t2_p40_h60_valid.npz
2026-04-21 20:38:23,165 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-21 20:38:23,385 | INFO | Loaded: windows_t2_p4


WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p40_h60
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler


Como acceder a las ventanas X e y:

```python
bundle_p40_h30 = bundles_L30["t2_p40_h30"]

X_train = bundle_p40_h30["train"]["X"]
y_train = bundle_p40_h30["train"]["y"]

X_valid = bundle_p40_h30["valid"]["X"]
y_valid = bundle_p40_h30["valid"]["y"]

X_test = bundle_p40_h30["test"]["X"]
y_test = bundle_p40_h30["test"]["y"]

scaler = bundle_p40_h30["scaler"]

print("Target :", bundle_p40_h30["target"])
print("Horizon:", bundle_p40_h30["horizon"])
print("Train  :", X_train.shape, y_train.shape)
print("Valid  :", X_valid.shape, y_valid.shape)
print("Test   :", X_test.shape, y_test.shape)
print("Scaler :", type(scaler).__name__)
```




In [10]:
bundles_L30

{'t2_p40_h30': {'window_size': 30,
  'target': 't2_p40_h30',
  'horizon': 30,
  'paths': {'train': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_train.npz',
   'valid': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_valid.npz',
   'test': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_test.npz',
   'scaler': '/content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl'},
  'scaler': StandardScaler(),
  'train': {'X': array([[[-0.11680385, -0.06583842, -0.2238865 , ..., -0.04129868,
            -1.507039  , -0.02778307],
           [ 0.0909589 ,  0.05818945, -0.07248875, ...,  0.3872164 ,
            -1.3723946 ,  0.08631181],
           [-0.18229802, -0.13053213, -0.21077001, ..., -0.1151728 ,
            -1.2194692 ,  0.02307655],
           ...,
           [ 0.40151003,  0.28442496,  0.59289074, ...,  0.47497907,
            -1.0690751 , -0.19294474],
         

### **4.5. Preparación de inputs según el tipo de modelo**

In [11]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

Estos sanity checks sirven para verificar, antes de entrenar, que los datos cargados tengan la estructura correcta y no vengan con errores silenciosos.

En concreto, comprueban que:

- X tenga el formato esperado: 3D (n, seq_len, n_features) o 2D (n, d_flat)
- y tenga forma válida para clasificación seq2one
- X e y tengan la misma cantidad de muestras
- no haya NaN ni inf
- las dimensiones sean consistentes entre train, valid y test
- exista más de una clase en y

Nos conviene tenerlos, porque ayudan a detectar errores de shape o de datos antes de llegar al entrenamiento.

In [12]:
from __future__ import annotations

from typing import Any, Optional, Tuple, Dict, Mapping
import numpy as np


# ============================================================
# SANITY CHECKS PARA DATASETS SEQ2ONE (T2)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía
    ni contenga NaN/inf.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(
    y: np.ndarray,
    *,
    name: str = "y",
    allow_seq_inputs_take_last: bool = False,
) -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)
    - (n, seq_len)       si allow_seq_inputs_take_last=True
    - (n, seq_len, 1)    si allow_seq_inputs_take_last=True
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    if allow_seq_inputs_take_last:
        if y.ndim == 2 and y.shape[1] > 1:
            return y[:, -1]

        if y.ndim == 3 and y.shape[2] == 1:
            return y[:, -1, 0]

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1)"
        f"{' o secuencial si allow_seq_inputs_take_last=True' if allow_seq_inputs_take_last else ''}. "
        f"Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )


def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)
      - y secuencial, opcionalmente, tomando el último valor

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: no aplica directamente.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D.
    allow_seq_inputs_take_last:
        Si y viene como secuencia, toma el último valor.
    """
    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    y = _normalize_y_seq2one(
        y,
        name=f"y[{split_name}]",
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
    )

    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info


def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_p40_h30",
      "horizon": 30,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Si no se pasan expected_*, usa TRAIN como referencia.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_bundles_seq2one(
    bundles: Mapping[str, Dict[str, Any]],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para todos los bundles de un diccionario:

    bundles[target] -> bundle
    """
    results = {}

    for target, bundle in bundles.items():
        results[target] = run_sanity_checks_for_bundle_seq2one(
            bundle,
            tag=target,
            verbose=verbose,
        )

    return results

In [13]:
sanity_results = run_sanity_checks_all_bundles_seq2one(
    bundles_L30,
    verbose=True,
)

[sanity_check_seq2one] train_t2_p40_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h30 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h30 | target=t2_p40_h30 | horizon=30
[sanity_check_seq2one] train_t2_p40_h60 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h60 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h60 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h60 | target=t2_p40_h60 | horizon=60
[sanity_check_seq2one] train_t2_p50_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p50_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | c

## **6. Módulo de métricas T2**

In [14]:
# ================================
# Setup para importar módulos del proyecto
# ================================

import sys
import importlib

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

# asegurar que metrics es paquete
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

from metrics.classification_probabilities import (
    compute_probabilistic_outputs,
    apply_decision_rule,
)

print("Módulos importados correctamente")

Módulos importados correctamente


In [15]:
# ================================
# Utilidades: outputs -> DataFrame
# ================================

from __future__ import annotations

import pandas as pd
from typing import Any


def classification_metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output de compute_classification_metrics(...)
    en una fila de DataFrame.

    Usa metrics_to_flat_dict(...) para aplanar la salida
    del módulo classification_metrics y luego agrega metadata
    del experimento.
    """
    flat_metrics = metrics_to_flat_dict(metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_metrics,
    }

    return pd.DataFrame([row])


def _flatten_dict(
    d: dict[str, Any],
    *,
    parent_key: str = "",
    sep: str = "_",
) -> dict[str, Any]:
    """
    Aplana un diccionario arbitrario de forma recursiva.

    Ejemplo:
    {"a": {"b": 1}} -> {"a_b": 1}
    """
    items: dict[str, Any] = {}

    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)

        if isinstance(v, dict):
            items.update(_flatten_dict(v, parent_key=new_key, sep=sep))
        else:
            items[new_key] = v

    return items


def probabilities_metrics_to_df(
    prob_metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output del módulo classification_probabilities
    en una fila de DataFrame.

    Como la estructura puede variar según la implementación,
    se aplana recursivamente el diccionario y luego se agrega
    metadata del experimento.
    """
    flat_prob_metrics = _flatten_dict(prob_metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_prob_metrics,
    }

    return pd.DataFrame([row])


logger.info("Utilidades de exportación a DataFrame cargadas")

2026-04-21 20:38:25,903 | INFO | Utilidades de exportación a DataFrame cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [16]:
# ================================
# Persistencia de métricas de clasificación
# ================================

from pathlib import Path
import pandas as pd


def load_classification_metrics_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics" / "classification_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen métricas previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics" / "classification_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"
    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path


# ================================
# Persistencia de probabilidades / decisión
# ================================

def load_classification_probabilities_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics" / "classification_probabilities",
) -> pd.DataFrame:
    """
    Carga resultados probabilísticos si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando probabilidades desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen probabilidades previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_probabilities(
    df_probabilities: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics" / "classification_probabilities",
) -> Path:
    """
    Guarda un DataFrame de probabilidades / decisión en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"
    df_probabilities.to_parquet(out_path, index=False)

    logger.info(f"Probabilidades guardadas en: {out_path}")

    return out_path

Ejemplo de uso con logistic regression:

```python
df_metrics_all = load_classification_metrics_if_exists(
    model_name="logistic_regression",
    split="valid",
)

df_probabilities_all = load_classification_probabilities_if_exists(
    model_name="logistic_regression",
    split="valid",
)
```

Guardar:
```python
save_classification_metrics(
    df_metrics_all,
    model_name="logistic_regression",
    split="valid",
)

save_classification_probabilities(
    df_probabilities_all,
    model_name="logistic_regression",
    split="valid",
)
```

## **8. Gestión de dispositivo y memoria**

In [17]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [18]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-21 20:38:29,059 | INFO | Seeds fijadas en 42


# **10. Entrenamiento de modelo**

## **10.1. Función unitaria por bundle**

In [19]:
def run_logistic_for_bundle_seq2one(
    bundle,
    *,
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000,
    C=1.0,
    random_state=42,
    input_mode="2d_flat",
    class_weight=None,
    verbose=False,
):
    """
    Ejecuta Logistic Regression para un bundle seq2one.
    Evalúa SOLO sobre VALID.
    """

    # =========================
    # 1. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    target = bundle.get("target")
    horizon = bundle.get("horizon")
    window_size = bundle.get("window_size")

    # =========================
    # 2. PREPARAR INPUT
    # =========================
    X_train_model = prepare_X_for_model(X_train, input_mode=input_mode)
    X_valid_model = prepare_X_for_model(X_valid, input_mode=input_mode)

    # =========================
    # 3. MODELO
    # =========================
    model = LogisticRegression(
        multi_class=multi_class,
        solver=solver,
        max_iter=max_iter,
        C=C,
        random_state=random_state,
        class_weight=class_weight,
    )

    # =========================
    # 4. TRAIN
    # =========================
    model.fit(X_train_model, y_train)

    # =========================
    # 5. PREDICT (VALID)
    # =========================
    y_pred_valid = model.predict(X_valid_model)
    y_proba_valid = model.predict_proba(X_valid_model)

    if verbose:
        print(
            f"[LOGISTIC] target={target} | horizon={horizon} | "
            f"window_size={window_size} | "
            f"X_train={X_train_model.shape} | X_valid={X_valid_model.shape}"
        )

    return {
        "model_name": "logistic_regression",
        "target": target,
        "horizon": horizon,
        "window_size": window_size,
        "input_mode": input_mode,
        "class_weight": class_weight,
        "model": model,
        "y_valid": y_valid,
        "y_pred_valid": y_pred_valid,
        "y_proba_valid": y_proba_valid,
    }

## **10.2. Función de evaluación sobre uno o más bundles**

In [39]:
from typing import Any, Dict, List, Sequence, Union
import pandas as pd


def eval_logistic_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    model_name: str = "logistic_regression",
    max_iter: int = 1000,
    random_state: int = 42,
    C: float = 1.0,
    multi_class: str = "multinomial",
    solver: str = "lbfgs",
    input_mode: str = "2d_flat",
    class_weight=None,
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
    verbose: bool = False,
) -> Dict[str, pd.DataFrame]:
    """
    Evalúa Logistic Regression para uno o varios bundles seq2one
    usando SOLO el split VALID.

    Retorna
    -------
    dict con:
    - "metrics": DataFrame de métricas de clasificación
    - "probabilities": DataFrame de métricas probabilísticas / decisión
    """

    # --------------------------------------------------
    # 1) Normalizar entrada
    # --------------------------------------------------
    if isinstance(bundles, dict):
        if "train" in bundles and "valid" in bundles:
            bundles_list: List[Dict[str, Any]] = [bundles]
        else:
            bundles_list = list(bundles.values())
    else:
        bundles_list = list(bundles)

    metrics_rows = []
    probabilities_rows = []

    # --------------------------------------------------
    # 2) Iterar por bundles
    # --------------------------------------------------
    for bundle in bundles_list:
        target = bundle.get("target", None)
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"model={model_name} | "
                f"class_weight={class_weight}"
            )

        # ----------------------------------------------
        # 3) Entrenar + predecir SOLO VALID
        # ----------------------------------------------
        preds = run_logistic_for_bundle_seq2one(
            bundle,
            multi_class=multi_class,
            solver=solver,
            max_iter=max_iter,
            C=C,
            random_state=random_state,
            input_mode=input_mode,
            class_weight=class_weight,
            verbose=False,
        )

        y_true = preds["y_valid"]
        y_pred = preds["y_pred_valid"]
        y_proba = preds["y_proba_valid"]

        # ----------------------------------------------
        # 4) Métricas de clasificación
        # ----------------------------------------------
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split="valid",
            target=target,
            labels=[-1, 0, 1],
        )

        df_metrics_row = classification_metrics_to_df(
            metrics,
            model=model_name,
            split="valid",
            window_size=window_size,
            target=target,
            horizon=horizon,
        )

        df_metrics_row["class_weight_mode"] = (
            "balanced" if class_weight == "balanced" else "none"
        )
        df_metrics_row["input_mode"] = input_mode
        df_metrics_row["C"] = C
        df_metrics_row["max_iter"] = max_iter
        df_metrics_row["multi_class"] = multi_class
        df_metrics_row["solver"] = solver

        metrics_rows.append(df_metrics_row)

        # ----------------------------------------------
        # 5) Outputs probabilísticos
        # ----------------------------------------------
        proba_df = compute_probabilistic_outputs(
            y_proba=y_proba,
            class_labels=[-1, 0, 1],
            y_true=y_true,
        )

        decision_df = apply_decision_rule(
            proba_df,
            long_class=1,
            short_class=-1,
            long_threshold=prob_threshold_long,
            short_threshold=prob_threshold_short,
        )

        # unir ambas tablas completas
        overlap_cols = [c for c in decision_df.columns if c in proba_df.columns]
        if overlap_cols:
            decision_df = decision_df.drop(columns=overlap_cols)

        df_prob = pd.concat(
            [proba_df.reset_index(drop=True), decision_df.reset_index(drop=True)],
            axis=1,
        )

        # metadata del experimento
        df_prob["model"] = model_name
        df_prob["split"] = "valid"
        df_prob["window_size"] = window_size
        df_prob["target"] = target
        df_prob["horizon"] = horizon
        df_prob["class_weight_mode"] = (
            "balanced" if class_weight == "balanced" else "none"
        )
        df_prob["input_mode"] = input_mode
        df_prob["C"] = C
        df_prob["max_iter"] = max_iter
        df_prob["multi_class"] = multi_class
        df_prob["solver"] = solver
        df_prob["threshold_long"] = prob_threshold_long
        df_prob["threshold_short"] = prob_threshold_short

        probabilities_rows.append(df_prob)

    # --------------------------------------------------
    # 6) Consolidar salida
    # --------------------------------------------------
    df_metrics_all = pd.concat(metrics_rows, ignore_index=True)
    df_probabilities_all = pd.concat(probabilities_rows, ignore_index=True)

    return {
        "metrics": df_metrics_all,
        "probabilities": df_probabilities_all,
    }

Cómo se usaría, para ambos bundles de una ventana:

```python
results_logistic = eval_logistic_bundles(
    bundles=bundle_L30,
    model_name="logistic_regression",
    max_iter=1000,
    random_state=SEED,
    C=1.0,
    multi_class="multinomial",
    solver="lbfgs",
    input_mode="2d_flat",
    class_weight=None,
    prob_threshold_long=0.40,
    prob_threshold_short=0.40,
    verbose=True,
)

df_metrics_all = results_logistic["metrics"]
df_probabilities_all = results_logistic["probabilities"]
```

## **10.3. Función orquestadora por `window_size`**

In [40]:
import gc
import pandas as pd


def run_logistic(
    window_size: int,
    *,
    targets: list[str] = TARGETS,
    verbose: bool = True,
    model_name: str = "logistic_regression",
    max_iter: int = 1000,
    random_state: int = 42,
    C: float = 1.0,
    multi_class: str = "multinomial",
    solver: str = "lbfgs",
    input_mode: str = "2d_flat",
    class_weight=None,
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
) -> dict[str, pd.DataFrame]:
    """
    Ejecuta Logistic Regression para una sola window_size
    sobre los targets T2 indicados.

    Evalúa SOLO sobre VALID.

    Retorna
    -------
    dict con:
    - "metrics": DataFrame consolidado de métricas de clasificación
    - "probabilities": DataFrame consolidado de métricas probabilísticas / decisión
    """

    size = int(window_size)

    bundles = None
    results = None

    model_name_effective = (
        f"{model_name}_balanced" if class_weight == "balanced" else model_name
    )

    try:
        # --------------------------------------------------
        # 1) Encabezado
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"LOGISTIC REGRESSION | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"targets       = {targets}")
            print(f"class_weight  = {class_weight}")
            print(f"input_mode    = {input_mode}")
            print(f"C             = {C}")
            print(f"max_iter      = {max_iter}")
            print(f"solver        = {solver}")
            print(f"multi_class   = {multi_class}")
            print(f"thr_long      = {prob_threshold_long}")
            print(f"thr_short     = {prob_threshold_short}")

        # --------------------------------------------------
        # 2) Construcción de bundles
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size} | n_targets={len(targets)}")

        bundles = create_bundles(
            window_size=size,
            targets=targets,
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        # --------------------------------------------------
        # 3) Evaluación SOLO VALID
        # --------------------------------------------------
        if verbose:
            print(
                f"\n[EVAL] L{size} | split=valid | "
                f"model={model_name_effective} | class_weight={class_weight}"
            )

        results = eval_logistic_bundles(
            bundles=bundles,
            model_name=model_name_effective,
            max_iter=max_iter,
            random_state=random_state,
            C=C,
            multi_class=multi_class,
            solver=solver,
            input_mode=input_mode,
            class_weight=class_weight,
            prob_threshold_long=prob_threshold_long,
            prob_threshold_short=prob_threshold_short,
            verbose=verbose,
        )

        df_metrics = (
            results["metrics"]
            .sort_values(["window_size", "target", "split", "horizon", "model"])
            .reset_index(drop=True)
        )

        df_probabilities = (
            results["probabilities"]
            .sort_values(["window_size", "target", "split", "horizon", "model"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 4) Resumen final
        # --------------------------------------------------
        if verbose:
            print(f"\n[DONE] L{size} | metrics_rows={len(df_metrics)} | probabilities_rows={len(df_probabilities)}")

            print("\n[METRICS]")
            cols_metrics = [
                c for c in [
                    "window_size",
                    "split",
                    "target",
                    "model",
                    "horizon",
                    "class_weight_mode",
                    "balanced_accuracy",
                    "f1_macro",
                ] if c in df_metrics.columns
            ]
            if cols_metrics:
                print(df_metrics[cols_metrics].to_string(index=False))

            print("\n[PROBABILITIES]")
            cols_probs = [
                c for c in [
                    "window_size",
                    "split",
                    "target",
                    "model",
                    "horizon",
                    "class_weight_mode",
                    "threshold_long",
                    "threshold_short",
                ] if c in df_probabilities.columns
            ]
            if cols_probs:
                print(df_probabilities[cols_probs].to_string(index=False))

        return {
            "metrics": df_metrics,
            "probabilities": df_probabilities,
        }

    finally:
        # --------------------------------------------------
        # 5) Liberación de memoria
        # --------------------------------------------------
        del bundles, results
        gc.collect()

Como se usa:



```python
results_logistic = run_logistic(
    window_size=30,
    targets=["t2_p40_h30", "t2_p40_h60", "t2_p50_h30"],
    model_name="logistic_regression",
    max_iter=1000,
    random_state=SEED,
    C=1.0,
    multi_class="multinomial",
    solver="lbfgs",
    input_mode="2d_flat",
    class_weight=None,
    prob_threshold_long=0.40,
    prob_threshold_short=0.40,
    verbose=True,
)

df_metrics_all = results_logistic["metrics"]
df_probabilities_all = results_logistic["probabilities"]
```



## **10.5. Ejecución final del experimento**

In [41]:
from sklearn.linear_model import LogisticRegression

results_logistic = run_logistic(
    window_size=30,
    targets=["t2_p40_h30", "t2_p40_h60", "t2_p50_h30"],
    model_name="logistic_regression",
    max_iter=1000,
    random_state=SEED,
    C=1.0,
    multi_class="multinomial",
    solver="lbfgs",
    input_mode="2d_flat",
    class_weight="balanced",  # 👈 ESTE ES EL CAMBIO
    prob_threshold_long=0.40,
    prob_threshold_short=0.40,
    verbose=True,
)

df_metrics_all = results_logistic["metrics"]
df_probabilities_all = results_logistic["probabilities"]

2026-04-21 21:00:05,903 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-21 21:00:05,904 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-21 21:00:05,924 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-21 21:00:05,925 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-21 21:00:05,945 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-21 21:00:05,945 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-21 21:00:05,949 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-21 21:00:05,950 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)



LOGISTIC REGRESSION | T2 SEQ2ONE | WINDOW_SIZE=L30
targets       = ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
class_weight  = balanced
input_mode    = 2d_flat
C             = 1.0
max_iter      = 1000
solver        = lbfgs
multi_class   = multinomial
thr_long      = 0.4
thr_short     = 0.4

[BUILD] L30 | n_targets=3


2026-04-21 21:00:06,033 | INFO | Loaded: windows_t2_p40_h60_train.npz
2026-04-21 21:00:06,034 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-21 21:00:06,053 | INFO | Loaded: windows_t2_p40_h60_valid.npz
2026-04-21 21:00:06,054 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-21 21:00:06,074 | INFO | Loaded: windows_t2_p40_h60_test.npz
2026-04-21 21:00:06,075 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-21 21:00:06,079 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-21 21:00:06,080 | INFO | Bundle cargado | target=t2_p40_h60 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-21 21:00:06,161 | INFO | Loaded: windows_t2_p50_h30_train.npz
2026-04-21 21:00:06,162 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-21 21:00:06,182 | INFO | Loaded: windows_t2_p50_h30_valid.npz
2026-04-21 21:00:06,183 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-21 21:00:06,202 | INFO | Loaded: windows_t2_p5

Se han truncado las últimas 5000 líneas del flujo de salida.
          30 valid t2_p50_h30 logistic_regression_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 logistic_regression_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 logistic_regression_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 logistic_regression_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 logistic_regression_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 logistic_regression_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 logistic_regression_balanced       30          balanced             0.4              0.4
          30 valid t2_p50_h30 logistic_regression_balanced       30          balanced             0.4     

## 10.6. Métricas

In [42]:
print('df_metrics_all')
df_metrics_all

df_metrics_all


,model,split,window_size,target,horizon,n_samples,accuracy,balanced_accuracy,f1_macro,f1_weighted,...,recall_weighted_naive,balanced_accuracy_gain_vs_naive,f1_macro_gain_vs_naive,f1_weighted_gain_vs_naive,class_weight_mode,input_mode,C,max_iter,multi_class,solver
0,logistic_regression_balanced,valid,30,t2_p40_h30,30,6882,0.374164,0.396396,0.366836,0.360868,...,0.393636,0.063063,0.178535,0.138501,balanced,2d_flat,1.0,1000,multinomial,lbfgs
1,logistic_regression_balanced,valid,30,t2_p40_h60,60,6882,0.407004,0.395298,0.381194,0.387120,...,0.363412,0.061964,0.203497,0.193388,balanced,2d_flat,1.0,1000,multinomial,lbfgs
2,logistic_regression_balanced,valid,30,t2_p50_h30,30,6882,0.412090,0.402972,0.388331,0.393041,...,0.359489,0.069639,0.212045,0.202922,balanced,2d_flat,1.0,1000,multinomial,lbfgs


In [43]:
print('df_probabilities_all')
df_probabilities_all

df_probabilities_all


,proba_-1,proba_0,proba_1,pred_label,confidence,y_true,is_correct,signal_raw,trade,model,...,target,horizon,class_weight_mode,input_mode,C,max_iter,multi_class,solver,threshold_long,threshold_short
0,0.185361,0.578971,0.235669,0,0.578971,1,False,0,False,logistic_regression_balanced,...,t2_p40_h30,30,balanced,2d_flat,1.0,1000,multinomial,lbfgs,0.4,0.4
1,0.220408,0.484428,0.295164,0,0.484428,1,False,0,False,logistic_regression_balanced,...,t2_p40_h30,30,balanced,2d_flat,1.0,1000,multinomial,lbfgs,0.4,0.4
2,0.228723,0.476908,0.294368,0,0.476908,1,False,0,False,logistic_regression_balanced,...,t2_p40_h30,30,balanced,2d_flat,1.0,1000,multinomial,lbfgs,0.4,0.4
3,0.208944,0.493380,0.297675,0,0.493380,1,False,0,False,logistic_regression_balanced,...,t2_p40_h30,30,balanced,2d_flat,1.0,1000,multinomial,lbfgs,0.4,0.4
4,0.201011,0.548830,0.250159,0,0.548830,1,False,0,False,logistic_regression_balanced,...,t2_p40_h30,30,balanced,2d_flat,1.0,1000,multinomial,lbfgs,0.4,0.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20641,0.273068,0.481182,0.245750,0,0.481182,0,True,0,False,logistic_regression_balanced,...,t2_p50_h30,30,balanced,2d_flat,1.0,1000,multinomial,lbfgs,0.4,0.4
20642,0.263822,0.474166,0.262012,0,0.474166,0,True,0,False,logistic_regression_balanced,...,t2_p50_h30,30,balanced,2d_flat,1.0,1000,multinomial,lbfgs,0.4,0.4
20643,0.277724,0.406309,0.315967,0,0.406309,0,True,0,False,logistic_regression_balanced,...,t2_p50_h30,30,balanced,2d_flat,1.0,1000,multinomial,lbfgs,0.4,0.4
20644,0.261694,0.401401,0.336905,0,0.401401,0,True,0,False,logistic_regression_balanced,...,t2_p50_h30,30,balanced,2d_flat,1.0,1000,multinomial,lbfgs,0.4,0.4


### Guardado de métricas

In [47]:
save_classification_metrics(
    df_metrics_all,
    model_name="logistic_regression",
    split="valid",
)

save_classification_probabilities(
    df_probabilities_all,
    model_name="logistic_regression",
    split="valid",
)

2026-04-21 21:14:47,888 | INFO | Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_metrics_logistic_regression_valid.parquet
2026-04-21 21:14:47,953 | INFO | Probabilidades guardadas en: /content/drive/MyDrive/neural_profit/metrics/classification_probabilities/classification_probabilities_logistic_regression_valid.parquet


PosixPath('/content/drive/MyDrive/neural_profit/metrics/classification_probabilities/classification_probabilities_logistic_regression_valid.parquet')

In [45]:
# =========================================
# Análisis rápido baseline Logistic
# =========================================

import pandas as pd

# -----------------------------
# 1) Ranking clasificación
# -----------------------------
print("\n" + "="*80)
print("RANKING CLASIFICACIÓN")
print("="*80)

cols_cls = [
    "target",
    "horizon",
    "accuracy",
    "balanced_accuracy",
    "balanced_accuracy_gain_vs_naive",
    "f1_macro",
    "f1_weighted",
]

print(
    df_metrics_all[cols_cls]
    .sort_values("balanced_accuracy", ascending=False)
    .to_string(index=False)
)

# -----------------------------
# 2) Resumen operativo por target
# -----------------------------
print("\n" + "="*80)
print("RESUMEN OPERATIVO POR TARGET")
print("="*80)

summary_probs = (
    df_probabilities_all
    .groupby(["target", "horizon"], as_index=False)
    .agg(
        n_obs=("target", "size"),
        trade_rate=("trade", "mean"),
        confidence_mean=("confidence", "mean"),
        confidence_median=("confidence", "median"),
        pct_pred_short=("pred_label", lambda s: (s == -1).mean()),
        pct_pred_flat=("pred_label", lambda s: (s == 0).mean()),
        pct_pred_long=("pred_label", lambda s: (s == 1).mean()),
        pct_signal_short=("signal_raw", lambda s: (s == -1).mean()),
        pct_signal_flat=("signal_raw", lambda s: (s == 0).mean()),
        pct_signal_long=("signal_raw", lambda s: (s == 1).mean()),
    )
)

print(summary_probs.to_string(index=False))

# -----------------------------
# 3) Precisión solo cuando hay trade
# -----------------------------
print("\n" + "="*80)
print("PRECISIÓN SOLO EN TRADES")
print("="*80)

trade_only = df_probabilities_all[df_probabilities_all["trade"] == True].copy()

if len(trade_only) == 0:
    print("No hubo trades con los thresholds actuales.")
else:
    trade_summary = (
        trade_only
        .groupby(["target", "horizon"], as_index=False)
        .agg(
            n_trades=("trade", "size"),
            precision_trades=("is_correct", "mean"),
            confidence_mean_trade=("confidence", "mean"),
            pct_trade_short=("signal_raw", lambda s: (s == -1).mean()),
            pct_trade_long=("signal_raw", lambda s: (s == 1).mean()),
        )
    )
    print(trade_summary.to_string(index=False))

# -----------------------------
# 4) Conclusión automática
# -----------------------------
print("\n" + "="*80)
print("CONCLUSIÓN RÁPIDA")
print("="*80)

best_target = (
    df_metrics_all.sort_values("balanced_accuracy", ascending=False)
    .iloc[0]["target"]
)

print(f"Mejor target por balanced_accuracy: {best_target}")

avg_trade_rate = df_probabilities_all["trade"].mean()
print(f"Trade rate global: {avg_trade_rate:.4f}")

if avg_trade_rate < 0.02:
    print("Diagnóstico: el modelo está siendo muy conservador con los thresholds actuales.")
elif avg_trade_rate < 0.10:
    print("Diagnóstico: el modelo opera poco; conviene revisar thresholds.")
else:
    print("Diagnóstico: el modelo sí genera una cantidad razonable de señales.")


RANKING CLASIFICACIÓN
    target  horizon  accuracy  balanced_accuracy  balanced_accuracy_gain_vs_naive  f1_macro  f1_weighted
t2_p50_h30       30  0.412090           0.402972                         0.069639  0.388331     0.393041
t2_p40_h30       30  0.374164           0.396396                         0.063063  0.366836     0.360868
t2_p40_h60       60  0.407004           0.395298                         0.061964  0.381194     0.387120

RESUMEN OPERATIVO POR TARGET
    target  horizon  n_obs  trade_rate  confidence_mean  confidence_median  pct_pred_short  pct_pred_flat  pct_pred_long  pct_signal_short  pct_signal_flat  pct_signal_long
t2_p40_h30       30   6882    0.240046         0.429001           0.416960        0.229149       0.542284       0.228567          0.112177         0.759954         0.127870
t2_p40_h60       60   6882    0.264022         0.439998           0.427009        0.206481       0.562191       0.231328          0.112613         0.735978         0.151409
t2_p50_h